# Data API Sender (Dictionary Version)

This notebook implements a system for:
1. Reading a pickle file containing a dictionary of text embeddings
2. Reading a csv file with text content and metadata for the text snippets
2. Processing the data
3. Sending it to a REST API


## Setup

### Imports

In [1]:
import pickle
import json
import jsonlines
import requests
import urllib
import bertopic
from typing import Any, Dict, List
import logging
from pathlib import Path
from time import sleep
from tqdm.notebook import tqdm
import numpy as np
from dotenv import load_dotenv
import os

### Configuration

In [ ]:
load_dotenv()

# Configuration variables
PICKLE_PATH = "./la_openai_embeddings_20240613.pkl"
PICKLE_OTHERINFO_PATH = "./la_tm_20240613.pkl"
CSV_OTHERINFO_PATH = "./la_paragraphs_20240520.csv"
CSV_TM_PATH = "./la_tm_20240613.csv"
SAVE_JSONL_PATH = "./la_openai_embeddings_20240613.jsonl"
SAVE_LOG_PATH = "./data_upload.log"

# API configuration
API_USER = "sal"
API_PROJECT = "sal-openai-large"
API_PROJECT_ID = 1
API_ENDPOINT = "http://localhost:8880/v1/embeddings/" + API_USER + "/" + API_PROJECT
API_LLM_SERVICE = "openai-large"
# Get API key from environment variable
API_KEY = os.getenv("DHAMPS_VDB_API_KEY")

HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}
BATCH_SIZE = None # Set to None to disable batching
RETRY_ATTEMPTS = 3
DELAY_BETWEEN_REQUESTS = 0.2
VERBOSE = False

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('data_upload.log'),
        logging.StreamHandler()
    ]
)

## Define Dict-to-JSON Saving Function

First, let's create a function to save a few samples in a readable JSON format:

In [3]:
def save_dict_to_json(data_dict: Dict, filepath: str, num_samples: int = 3):
    """Save a sample of the dictionary data to JSON."""
    
    # Create a sample dictionary with first num_samples items
    sample_dict = dict(list(data_dict.items())[:num_samples])
    
    # Convert numpy arrays to lists if present
    processed_dict = {}
    for key, value in sample_dict.items():
        if isinstance(value, dict):
            processed_dict[key] = {
                k: (v.tolist() if isinstance(v, np.ndarray) else v)
                for k, v in value.items()
            }
        else:
            processed_dict[key] = (
                value.tolist() if isinstance(value, np.ndarray) else value
            )
    
    # Save to JSON with nice formatting
    with open(filepath, 'w') as f:
        json.dump(processed_dict, f, indent=2)
    
    print(f"Saved {num_samples} samples to {filepath}")

## Load and Examine Data

### Embeddings

Load embeddings

In [4]:
# Load the pickle file with embeddings
with open(PICKLE_PATH, 'rb') as f:
    data_dict = pickle.load(f)

embeddings_dict = data_dict["text-embedding-3-small"]

# Display basic information about the dictionary
print(f"Number of items in embeddings dictionary: {len(embeddings_dict)}")

# Display structure of first item
first_key = next(iter(embeddings_dict))
print("\nStructure of first item:")
print(f"Key: {first_key}")
first_value = embeddings_dict[first_key]
print("Value keys:", list(first_value.keys()) if isinstance(first_value, dict) else type(first_value))
# what kind of numbers is in the value list?
print("Value type:", type(first_value[1]))

Number of items in embeddings dictionary: 65502

Structure of first item:
Key: https://id.salamanca.school/texts/W0001:vol1.1.1.2.1
Value keys: <class 'list'>
Value type: <class 'float'>


Save a sample

In [5]:

# Save a sample to JSON for inspection
save_dict_to_json(embeddings_dict, 'sample_data.json', num_samples=3)


Saved 3 samples to sample_data.json


### Other information

Load other information

In [6]:
# Load the csvs with other information
import pandas as pd
df_tm = pd.read_csv(CSV_TM_PATH)
df_otherinfo = pd.read_csv(CSV_OTHERINFO_PATH)

# print the first 3 rows of each dataframe
print(df_tm.head(3))
print(df_otherinfo.head(3))


   Topic  Count                                               Name  \
0     -1  27533  -1_Exclusion and Succession Laws in Legal and ...   
1      0   1614              0_Sacrament of Penance and Confession   
2      1   1581  1_clerical appointments and ecclesiastical aut...   

                                      Representation  \
0  ['Exclusion and Succession Laws in Legal and C...   
1            ['Sacrament of Penance and Confession']   
2  ['clerical appointments and ecclesiastical aut...   

                                        Cohere-Label  \
0  ['Legal and canonical exclusion and succession...   
1  ['The nature and mechanics of the Sacrament of...   
2  ['Topic name: Ecclesiastical appointments and ...   

                                         GPT-Summary  \
0  ['### Exclusion and Succession Laws in Legal a...   
1  ['The topic of "Sacrament of Penance and Confe...   
2  ['The topic outlined by the keywords "clerical...   

                                 Representat

## Build Sendable Data

### Build json objects as the API expects them

In [7]:
# Create a list to store the json objects
json_objects = []

# Iterate over the keys of the embeddings dictionary
for key, value in tqdm(list(embeddings_dict.items())):
    # Look up the row in the df_otherinfo dataframe that has the dict key in the url column
    row = df_otherinfo[df_otherinfo["url"] == key]
    
    # Check if the row was found
    if len(row) == 0:
        logging.warning(f"Could not find row in df_otherinfo for key: {key}")
        continue
    
    # Get the first row (should be unique, but make sure)
    row = row.iloc[0]
    
    # Check if all values in the vector are floats
    # for i, v in enumerate(value):
    #    if not isinstance(v, float):
    #        logging.warning(f"Non-float value found at index {i} in key {key}: {v} (type: {type(v)})")
    #        value[i] = float(v)  # Convert to float if necessary

    # Build the json object
    json_object = {
        "text_id": urllib.parse.quote(key, safe=''),
        "user_handle": API_USER,
        "project_handle": API_PROJECT,
        "project_id": API_PROJECT_ID,
        "llm_service_handle": API_LLM_SERVICE,
        "text": row["content"],
        "vector": value,
        "vector_dim": len(value),
        "metadata": {
            "url": row["url"],
            "xmlid": row["xmlid"],
            "wid": row["wid"],
            "author": row["author-name"],
            "author_id": row["author-id"],
            "year": int(row["year"]) if isinstance(row["year"], np.int64) else row["year"],
            "lang": row["lang"],
        }
    }
    
    # Append to the list
    json_objects.append(json_object)

print("Done. Number of json objects:", len(json_objects))

  0%|          | 0/65502 [00:00<?, ?it/s]

Done. Number of json objects: 65502


### Save to JSONL

Save sendable objects to a jsonl file

In [8]:
# Save the json objects to a jsonlines file
with jsonlines.open(SAVE_JSONL_PATH, 'w') as writer:
    writer.write_all(json_objects)

print(f"Saved {len(json_objects)} json objects to send_data.jsonl")


Saved 65502 json objects to send_data.jsonl


## Define class and functions to send a list of objects

In [9]:
class JsonListApiSender:
    def __init__(
        self,
        api_endpoint: str,
        headers: Dict[str, str] = None,
        retry_attempts: int = 3,
        delay_between_requests: float = 0.1,
        verbose: bool = True
    ):
        """
        Initialize the JSON List API Sender.
        
        Args:
            api_endpoint: URL to send POST requests to
            headers: Optional headers for the request
            retry_attempts: Number of times to retry a failed request
            delay_between_requests: Delay between each request
            verbose: Whether to print detailed logging
        """
        self.api_endpoint = API_ENDPOINT
        self.headers = HEADERS or {"Content-Type": "application/json"}
        self.retry_attempts = RETRY_ATTEMPTS
        self.delay_between_requests = DELAY_BETWEEN_REQUESTS
        self.verbose = VERBOSE
        self.logger = logging.getLogger(__name__)

    def send_single_item(self, item: Dict[str, Any]) -> bool:
        """
        Send a single JSON item to the API with retry logic.
        
        Args:
            item: JSON object to send
        
        Returns:
            Boolean indicating success or failure
        """
        for attempt in range(self.retry_attempts):
            try:
                response = requests.post(
                    self.api_endpoint,
                    json={"embeddings": [item]},
                    headers=self.headers
                )
                response.raise_for_status()
                
                if self.verbose:
                    self.logger.info(f"Successfully sent item with text_id: {item.get('text_id')}")
                
                return True
            
            except requests.exceptions.RequestException as e:
                # Log error only for the last attempt
                if attempt == self.retry_attempts - 1:
                    error = e.read() if hasattr(e, 'read') else str(e)
                    self.logger.error(f"Failed to send item with text_id: {item.get('text_id')}. Error: {str(error)}")
                
                # Exponential backoff
                sleep(2 ** attempt)
        
        return False

    def send_list(
        self, 
        json_list: List[Dict[str, Any]], 
        num_items: int = BATCH_SIZE
    ) -> Dict[str, int]:
        """
        Send a list of JSON objects to the API.
        
        Args:
            json_list: List of JSON objects to send
            num_items: Number of items to send (None = send all)
        
        Returns:
            Dictionary with counts of successful and failed items
        """
        # Determine number of items to send
        if num_items is None:
            num_items = len(json_list)
        else:
            num_items = min(num_items, len(json_list))
        
        # Slice the list to the desired number of items
        items_to_send = json_list[:num_items]
        
        # Tracking variables
        success_count = 0
        fail_count = 0
        failed_items = set()
        
        # Log start of sending process
        self.logger.info(f"Starting to send {num_items} items to {self.api_endpoint}")
        
        # Progress bar
        for item in tqdm(items_to_send, desc="Sending items", total=num_items):
            if self.send_single_item(item):
                success_count += 1
            else:
                fail_count += 1
                failed_items.add(item.get('text_id'))
            
            # Delay between requests
            sleep(self.delay_between_requests)
        
        # Log overall results
        self.logger.info(
            f"Sending complete. "
            f"Sent: {num_items}, "
            f"Successful: {success_count}, "
            f"Failed: {fail_count}"
        )
        
        return {
            "total_sent": num_items,
            "successful": success_count,
            "failed": fail_count,
            "failed_ids": list(failed_items)
        }

# Example usage:
# sender = JsonListApiSender(api_endpoint="https://your-api-endpoint.com", verbose=True)
# results = sender.send_list(your_json_list, num_items=10)

## Send Data to Vector Database

In [13]:
sender = JsonListApiSender(api_endpoint=API_ENDPOINT, verbose=VERBOSE)
results = sender.send_list(json_objects, num_items=None)


2024-12-08 04:46:34,901 - INFO - Starting to send 65502 items to http://localhost:8880/v1/embeddings/sal/sal-openai-large


Sending items:   0%|          | 0/65502 [00:00<?, ?it/s]

2024-12-08 11:44:42,876 - INFO - Sending complete. Sent: 65502, Successful: 65502, Failed: 0


## Report Results

In [11]:
# Display results
print(results)

{'total_sent': 1000, 'successful': 1000, 'failed': 0, 'failed_ids': []}
